In [ ]:
import scrapy
import pandas as pd
from scrapy.crawler import CrawlerProcess
from scrapy import Request
from scrapy.spiders import Spider
from datetime import datetime
import time

In [ ]:
from scrapy.utils.project import get_project_settings

settings = get_project_settings()

settings.set("COOKIES_ENABLED", True, priority="cmdline")
settings.set("ROBOTSTXT_OBEY", False, priority="cmdline")


In [ ]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")

In [ ]:
client = MongoDBClient(mongo_uri, db_name, collection)

In [ ]:
class LaRazonSpider(Spider):
    name = "larazon"
    allowed_domains = ["www.la-razon.com"]
    start_urls = [
        f"https://www.la-razon.com/tags/feminicidio/page/{i}/" for i in range(87, 0,-1)
    ]
    
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36 Edg/134.0.0.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0"
    ]
    
    custom_headers = {
                    "User-Agent": user_agents[0],
                    "Connection": 'keep_alive',
                    "Host": "www.la-razon.com",
                    "Accept-Encoding": "text/html",
                }
    
    avoid_sections = ["/lr-article/", "/mundo/", "/voces/", "/opinion/", "/la-revista/", "/politico/","/marcas/","/economia/"]
    
    def __init__(self):
        self.items = []
        self.mongo_client = client
        
    def date_formatter(self, url, date_format="%Y%m%d"):
        try:
            url_split = url.split("/")
            date_str = url_split[4] + url_split[5] + url_split[6]
            date_publish = datetime.strptime(date_str, date_format)
            return date_publish
        except Exception as e:
            self.logger.error(f"Error al formatear fecha: {e}")
            return None
    
    def tittle_formatter(self, title):
        try:
            title = title.replace("“", '"')
            title = title.replace("”", '"')
            return title
        except Exception as e:
            self.logger.error(f"Error al formatear título: {e}")
            return title
    
    def tag_formatter(self, tags):
        try:
            list_tags = [t.lower() for t in tags]
            return list_tags
        except Exception as e:
            self.logger.error(f"Error al formatear tags: {e}")
            return tags
    
    def section_formatter(self, url):
        try:
            url_split = url.split("/")
            section = url_split[3]
            return section
        except Exception as e:
            self.logger.error(f"Error al formatear sección: {e}")
            return url

    def body_formatter(self, body):
        try:
            new_body = [
                b.strip()
                .replace("\xa0", " ")
                .replace('\"', "")
                .replace("\ufeff", " ")
                .replace("“", '"')
                .replace("”", '"')
                .replace("\u200b", " ")
                for b in body
            ]
            new_body = [b for b in new_body if b != " "]
            return new_body
        except Exception as e:
            self.logger.error(f"Error al formatear cuerpo: {e}")
            return body

    def start_requests(self):
        for url in self.start_urls:
            self.logger.info(f"Enviando request a: {url}")
            yield Request(url=url, callback=self.parse_response, headers=self.custom_headers)
            
    
    def parse_response(self, response):
        self.logger.info(f"Recibida respuesta: {response.url}")
        try:
            noticias = response.xpath('(//div[@class="articles-list"])[1]//div[@class="article-meta "]/a/@href').getall()
            
            self.logger.info(f"Total noticias encontradas: {len(noticias)}")
            for noticia in noticias:
                if not any(s in noticia for s in self.avoid_sections):
                    self.logger.info(f"Enviando request a: {noticia}")
                    yield Request(url=noticia, callback=self.parse_news, headers=self.custom_headers)
        except Exception as e:
            self.logger.error(f"Error al procesar la respuesta JSON: {e}")
            return
    
    def parse_news(self, response):
        title = response.xpath("(//h1[@class='title'])[1]/text()").get()
        item = {}
        item["url"] = response.url
        item["title"] = self.tittle_formatter(title)
        tags = response.xpath('(//div[@class="lr-tags-cloud-block"])[1]//a[contains(@href, "tag")]/li/text()').getall()
        item["tags"] = self.tag_formatter(tags)
        item["section"] = self.section_formatter(response.url)
        body = [
            p.xpath("string(.)").get()
            for p in response.xpath("(//div[contains(@class, 'article-body')])[1]/p")
        ]
        item["body"] = self.body_formatter(body)
        item["date_published"] = self.date_formatter(response.url)
        item["source"] = "larazon"
        self.items.append(item)
        self.logger.info(f"Noticia agregada: {item['title']}")
        time.sleep(2)
    
    def close(self, reason):
        self.mongo_client.connect()
        self.logger.info("Guardando datos en MongoDB")
        for item in self.items:
            try:
                self.mongo_client.insert_new_document(item, "url")
            except Exception as e:
                self.logger.error(f"Error al insertar en MongoDB: {e}")

        self.logger.info(f"Spider cerrado por la razón: {reason}")

In [ ]:
process = CrawlerProcess(settings)
process.crawl(LaRazonSpider)
process.start()

DEBUG:pymongo.topology:{"topologyId": {"$oid": "67cfc4743e1ecf1b2cfd2a59"}, "driverConnectionId": 1, "serverConnectionId": 136279, "serverHost": "bolivia-data-shard-00-00.5b2cx.mongodb.net", "serverPort": 27017, "awaited": true, "durationMS": 9983.999999705702, "reply": "{\"topologyVersion\": {\"processId\": {\"$oid\": \"67c5ef507d399c43825333c2\"}, \"counter\": 4}, \"hosts\": [\"bolivia-data-shard-00-00.5b2cx.mongodb.net:27017\", \"bolivia-data-shard-00-01.5b2cx.mongodb.net:27017\", \"bolivia-data-shard-00-02.5b2cx.mongodb.net:27017\"], \"setName\": \"atlas-13auh3-shard-0\", \"setVersion\": 15, \"secondary\": true, \"primary\": \"bolivia-data-shard-00-02.5b2cx.mongodb.net:27017\", \"tags\": {\"provider\": \"AWS\", \"region\": \"SA_EAST_1\", \"diskState\": \"READY\", \"nodeType\": \"ELECTABLE\", \"availabilityZone\": \"sae1-az1\", \"workloadType\": \"OPERATIONAL\"}, \"me\": \"bolivia-data-shard-00-00.5b2cx.mongodb.net:27017\", \"lastWrite\": {\"opTime\": {\"ts\": {\"$timestamp\": {\"t\